# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [87]:
import warnings
warnings.filterwarnings('ignore')

In [88]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [5]:
#!pip install pandas

In [89]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [90]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


## LLMChain

In [ ]:
!pip install langchain_community

In [91]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [92]:
#Replace None by your own value and justify
# temperature=0 -> deterministic, factual output. For a product description we want
# a consistent, reproducible answer rather than a creative one, so 0 is a good choice.
# gpt-3.5-turbo (the library default) is retired, so we name a current model explicitly.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [93]:
prompt = ChatPromptTemplate.from_template(
    "Describe the {product} in a classic manner"
)

In [94]:

chain = LLMChain(llm=llm, prompt=prompt)

In [95]:
product = "Deodrant"
chain.run(product)

"Deodorant, a quintessential personal care product, serves as a guardian of freshness and an emblem of cleanliness in the daily rituals of grooming. Traditionally crafted to combat the unwelcome odors that arise from perspiration, it embodies a harmonious blend of science and artistry. \n\nIn its classic form, deodorant often presents itself in a sleek stick or a delicate roll-on, encased in a vessel that reflects both elegance and functionality. The formulation typically comprises a thoughtful amalgamation of fragrant essential oils, soothing agents, and antimicrobial compounds, all meticulously designed to neutralize odor-causing bacteria while imparting a subtle, pleasing aroma.\n\nThe application of deodorant is a ritualistic act, a moment of self-care that transcends mere necessity. As one glides the stick across the skin or gently rolls the applicator, there is a sense of assurance that accompanies the act—a promise of confidence and poise throughout the day. \n\nIn essence, deod

## SimpleSequentialChain

In [96]:
from langchain.chains import SimpleSequentialChain

In [97]:
# Higher temperature -> more creative taglines
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    #Repeat the initial query or create a new query that would feed into the second prompt
    template = "Create a short, catchy tagline for a {product}. Return only the tagline."
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [98]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    #Write the second prompt query that takes an input variable whose input will come from the previous prompt"
    # {tagline} receives the output of chain_one
    template = "Write a funny 3-sentence product description based on this tagline: {tagline}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [99]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [100]:
overall_simple_chain.run(product)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


"Stay Fresh, Feel Fierce!"
Introducing our revolutionary Stay Fresh, Feel Fierce deodorant—because why should your underarms be the only part of you that’s fresh while you’re slaying the day? With a scent so fierce, it’s like a pep talk in a stick, ready to tackle any awkward elevator ride or surprise dance-off. Embrace your inner warrior and let your confidence soar, while your underarms become the envy of every gym-goer and yoga mat!

> Finished chain.


'Introducing our revolutionary Stay Fresh, Feel Fierce deodorant—because why should your underarms be the only part of you that’s fresh while you’re slaying the day? With a scent so fierce, it’s like a pep talk in a stick, ready to tackle any awkward elevator ride or surprise dance-off. Embrace your inner warrior and let your confidence soar, while your underarms become the envy of every gym-goer and yoga mat!'

**Repeat the above twice for different products**

### Repeat 1 - different product

In [101]:
product = "Electric Toothbrush"
overall_simple_chain.run(product)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


"Brush Brighter, Smile Stronger!"
Introducing the ToothTornado 3000 – the only toothbrush that doubles as a mini workout! With its built-in resistance feature, you'll brush away plaque while flexing those biceps, giving you a dazzling smile and a stronger grip on your lunch! Say goodbye to weak brushes and hello to a grin that could lift weights – literally!

> Finished chain.


"Introducing the ToothTornado 3000 – the only toothbrush that doubles as a mini workout! With its built-in resistance feature, you'll brush away plaque while flexing those biceps, giving you a dazzling smile and a stronger grip on your lunch! Say goodbye to weak brushes and hello to a grin that could lift weights – literally!"

### Repeat 2 - different product

In [102]:
product = "Noise Cancelling Headphones"
overall_simple_chain.run(product)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


"Silence the world, tune into you."
Introducing the ultimate noise-canceling headphones that do more than just block out the blaring chaos of life—they give you the perfect excuse to ignore your in-laws at family gatherings! With our advanced sound isolation technology, you can finally hear your inner thoughts while pretending to be deeply engrossed in a podcast about the history of cheese. So go ahead, silence the world and tune into you—because sometimes, your best conversation partner is the only person who won’t interrupt you!

> Finished chain.


'Introducing the ultimate noise-canceling headphones that do more than just block out the blaring chaos of life—they give you the perfect excuse to ignore your in-laws at family gatherings! With our advanced sound isolation technology, you can finally hear your inner thoughts while pretending to be deeply engrossed in a podcast about the history of cheese. So go ahead, silence the world and tune into you—because sometimes, your best conversation partner is the only person who won’t interrupt you!'

## SequentialChain

In [103]:
from langchain.chains import SequentialChain

In [104]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)


first_prompt = ChatPromptTemplate.from_template(
  #This prompt should translate a review
  template = "Translate the {review} to kannada language"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="translated_review"
                    )


In [105]:
second_prompt = ChatPromptTemplate.from_template(
    #Write a promplt to summarize a review
    template = "Summarize the following review in 1 sentence:\n\n{review}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt,
                     output_key="summarized_review"
                    )

In [106]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template(
    template = "Translate the {summarized_review} to German"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="german_review"
                      )


In [107]:
# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a short, polite follow-up message from the seller to the customer who wrote this review.\n\n"
    "Review summary: {summarized_review}\n\n"
    "German version of the summary: {german_review}\n\n"
    "Write the follow-up message in German."
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )

In [108]:
# overall_chain: input= review
# and output= translated_review, summarized_review, german_review, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["review"],
    output_variables=["translated_review", "summarized_review", "german_review", "followup_message"],
    verbose=True
)

In [109]:
review = df.Review[5]
overall_chain(review)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.


{'review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\r\nVieux lot ou contrefaçon !?",
 'translated_review': 'The translation of the provided text to Kannada is:\n\n"ನಾನು ಇದರ ರುಚಿಯನ್ನು ಅಸಾಧಾರಣ ಎಂದುಕೊಂಡಿದ್ದೇನೆ. ಫೋಮ್ ಹಿಡಿದಿಟ್ಟುಕೊಳ್ಳುವುದಿಲ್ಲ, ಇದು ವಿಚಿತ್ರವಾಗಿದೆ. ನಾನು ವ್ಯಾಪಾರದಲ್ಲಿ ಒಂದೇ ಮಾದರಿಯವುಗಳನ್ನು ಖರೀದಿಸುತ್ತಿರುವೆ ಮತ್ತು ರುಚಿ ಇನ್ನಷ್ಟು ಉತ್ತಮವಾಗಿದೆ... ಹಳೆಯ ಕಟ್ಟಿ ಅಥವಾ ನಕಲಿ!? "',
 'summarized_review': "L'auteur trouve le goût médiocre et la texture insatisfaisante, suggérant une possible contrefaçon du produit par rapport à ceux achetés en magasin.",
 'german_review': 'Der Autor findet den Geschmack mittelmäßig und die Textur unbefriedigend, was auf eine mögliche Fälschung des Produkts im Vergleich zu den im Geschäft gekauften hinweist.',
 'followup_message': 'Liebe/r [Kundenname],\n\nvielen Dank für Ihr Feedback zu unserem Produkt. Es tut uns leid zu hören, dass der Geschmack und die Textur nicht Ihre

**Repeat the above twice for different products or reviews**

### Repeat 1 - different review

In [110]:
review = df.Review[0]
overall_chain(review)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.


{'review': 'I ordered a king size set. My only criticism would be that I wish seller would offer the king size set with 4 pillowcases. I separately ordered a two pack of pillowcases so I could have a total of four. When I saw the two packages, it looked like the color did not exactly match. Customer service was excellent about sending me two more pillowcases so I would have four that matched. Excellent! For the cost of these sheets, I am satisfied with the characteristics and coolness of the sheets.',
 'translated_review': 'ನಾನು ಕಿಂಗ್ ಗಾತ್ರದ ಸೆಟ್ ಅನ್ನು ಆರ್ಡರ್ ಮಾಡಿದೆ. ನನ್ನ ಏಕೈಕ ಟೀಕೆ ಏನೆಂದರೆ, ಕಿಂಗ್ ಗಾತ್ರದ ಸೆಟ್\u200c ಅನ್ನು ನಾಲ್ಕು ತಂಬುಗಳು ಜೊತೆಗೆ ನೀಡಬೇಕೆಂದೆನ್ ಬಯಸುತ್ತೇನೆ. ನಾನು ನಾಲ್ಕು ತಂಬುಗಳನ್ನು ಹೊಂದಿ ಹಂಚಿಕೊಳ್ಳಲು ಪ್ರತ್ಯೇಕವಾಗಿ ಎರಡು ತಂಬುಗಳ ಪ್ಯಾಕ್ ಅನ್ನು ಆರ್ಡರ್ ಮಾಡಿದೆ. ಎರಡು ಪ್ಯಾಕೇಜ್\u200cಗಳನ್ನು ನೋಡಲು, ಬಣ್ಣವು ಸಂಪೂರ್ಣವಾಗಿ ಹೊಂದಿಕೆಯಾಗದಂತೆ ಕಾಣಿಸುತ್ತಿತ್ತು. ನನ್ನೊಂದಿಗೆ ನಾಲ್ಕು ಹೊಂದಾಣಿಕೆಯ ತಂಬುಗಳನ್ನು ನೀಡಲು ಗ್ರಾಹಕ ಸೇವೆ ಅತ್ಯುತ್ತಮವಾಗಿತ್ತು. ಅತ್ಯುತ್ತಮ! ಈ ಶೀಟ್\u200cಗಳ ವೆಚ್ಚವನ್ನು ತಗೋಲಿದರೆ, ಶೀಟ್\u200cಗಳ ವೈಶಿಷ್ಟ್ಯಗಳ

### Repeat 2 - different review

In [111]:
review = df.Review[3]
overall_chain(review)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.


{'review': 'This is the best throw pillow fillers on Amazon. I’ve tried several others, and they’re all cheap and flat no matter how much fluffing you do. Once you toss these in the dryer after you remove them from the vacuum sealed shipping material, they fluff up great',
 'translated_review': 'ಇದು ಆಮೆಜಾನ್\u200cನಲ್ಲಿ ಉತ್ತಮ ತೋರ್ ಬಿಷ್ಟರ ಪುಷ್ಪಕಗಳು. ನಾನು ಇತರ ಹಲವಾರು ಪ್ರಯತ್ನಿಸಿದ್ದೇನೆ, ಮತ್ತು ಅವು ಎಲ್ಲವೂ ಕಡಿಮೆ ದರದ ಮತ್ತು ಸಮತೋಲಿತವಾಗಿವೆ, ನೀವು ಎಷ್ಟು ಶುಭ್ರಪಡಿಸಿದರೂ ಸಹ. ನೀವು ವಾಕ್ಯಮೋಡ್\u200cನ ಶಿಪ್ಪಿಂಗ್ ವಸ್ತುಗಳಿಂದ ಅವುಗಳನ್ನು ತೆಗೆದ ನಂತರ, ಈ ಪುಷ್ಪಕಗಳನ್ನು ಒಗ್ಗಿಸುವಾಗ, ಅವು ಉತ್ತಮವಾಗಿ ಫ್ಲಫ್ ಆಗುತ್ತವೆ.',
 'summarized_review': 'This reviewer highly recommends these throw pillow fillers as the best on Amazon, noting their superior fluffiness compared to other brands after being dried post-shipping.',
 'german_review': 'Dieser Rezensent empfiehlt diese Kissenfüllungen dringend als die besten auf Amazon und hebt ihre überlegene Flauschigkeit im Vergleich zu anderen Marken nach dem Trocknen nach dem Versand hervor.',


## Router Chain

In [112]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [113]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [114]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [115]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [120]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [117]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [121]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [122]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [123]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [124]:
chain.run("What is black body radiation?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


physics: {'input': 'What is black body radiation?'}
> Finished chain.


'Black body radiation refers to the electromagnetic radiation emitted by an idealized object known as a "black body." A black body is a perfect absorber of all incident radiation, meaning it absorbs all wavelengths of light and does not reflect any. When a black body is heated, it emits radiation across a continuous spectrum of wavelengths.\n\nThe key characteristics of black body radiation include:\n\n1. **Temperature Dependence**: The spectrum of radiation emitted by a black body depends solely on its temperature. As the temperature increases, the peak wavelength of the emitted radiation shifts to shorter wavelengths, a phenomenon described by Wien\'s Displacement Law.\n\n2. **Planck\'s Law**: The distribution of radiation emitted by a black body at a given temperature is described by Planck\'s Law, which provides the intensity of radiation at different wavelengths.\n\n3. **Stefan-Boltzmann Law**: The total energy radiated per unit surface area of a black body is proportional to the 

In [125]:
chain.run("what is 2 + 2")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


math: {'input': 'what is 2 + 2'}
> Finished chain.


'To solve the problem \\(2 + 2\\), we can break it down into its component parts:\n\n1. Identify the numbers involved: We have the numbers 2 and 2.\n2. Understand the operation: The operation we are performing is addition.\n\nNow, we can combine the two numbers:\n\n\\[\n2 + 2 = 4\n\\]\n\nSo, the answer to the question \\(2 + 2\\) is \\(4\\).'

In [126]:
chain.run("Why does every cell in our body contain DNA?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA serves as the fundamental blueprint for life. Here are several key reasons why this is the case:\n\n1. **Genetic Information Storage**: DNA (deoxyribonucleic acid) contains the genetic instructions necessary for the development, functioning, growth, and reproduction of all living organisms. It encodes the information required to produce proteins, which perform a vast array of functions within cells.\n\n2. **Cellular Function and Identity**: Each cell type in our body (e.g., muscle cells, nerve cells, skin cells) has a specific function and identity, which is determined by the expression of certain genes. While all cells contain the same DNA, different cells express different subsets of genes, leading to their specialized functions. This process is regulated by various factors, including transcription factors and epigenetic modifications.\n\n3. **Replication and Division**: When cells divide, they must replicate their DNA so that each dau

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [127]:
chain.run("Who was the first emperor of the Roman Empire and how did he come to power?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


History: {'input': 'Who was the first emperor of the Roman Empire and how did he come to power?'}
> Finished chain.


'The first emperor of the Roman Empire was Augustus, originally known as Gaius Octavius Thurinus. He came to power following a tumultuous period in Roman history marked by civil wars and political instability.\n\nAugustus was born on September 23, 63 BCE, and was the grandnephew and adopted son of Julius Caesar. After Caesar\'s assassination in 44 BCE, a power struggle ensued in Rome. Octavian (as he was known before taking the title Augustus) formed the Second Triumvirate in 43 BCE with Mark Antony and Marcus Lepidus to defeat Caesar\'s assassins and consolidate power.\n\nThe Triumvirate successfully defeated the forces of Brutus and Cassius at the Battle of Philippi in 42 BCE. However, tensions soon arose between Octavian and Antony, particularly after Antony\'s alliance with Cleopatra, the Queen of Egypt. This rivalry culminated in the naval Battle of Actium in 31 BCE, where Octavian\'s forces decisively defeated Antony and Cleopatra.\n\nFollowing their defeat, Antony and Cleopatra 

In [128]:
chain.run("What is the time complexity of binary search and why?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


computer science: {'input': 'What is the time complexity of binary search and why?'}
> Finished chain.


'The time complexity of binary search is \\( O(\\log n) \\), where \\( n \\) is the number of elements in the sorted array or list being searched.\n\n### Explanation:\n\n1. **Divide and Conquer**: Binary search operates on a sorted array by repeatedly dividing the search interval in half. It starts by comparing the target value to the middle element of the array.\n\n2. **Halving the Search Space**: If the target value is equal to the middle element, the search is complete. If the target value is less than the middle element, the search continues in the left half of the array; if it is greater, the search continues in the right half. Each comparison effectively halves the number of elements to be searched.\n\n3. **Logarithmic Growth**: Because the search space is halved with each step, the number of comparisons needed to find the target value grows logarithmically with respect to the number of elements. Specifically, if you have \\( n \\) elements, the maximum number of times you can ha

In [129]:
chain.run("How does photosynthesis convert sunlight into chemical energy?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


biology: {'input': 'How does photosynthesis convert sunlight into chemical energy?'}
> Finished chain.


'Photosynthesis is a complex biochemical process that converts light energy from the sun into chemical energy stored in glucose, a sugar molecule. This process primarily occurs in the chloroplasts of plant cells, as well as in some algae and certain bacteria. Photosynthesis can be divided into two main stages: the light-dependent reactions and the light-independent reactions (Calvin cycle).\n\n### 1. Light-Dependent Reactions\n\nThese reactions take place in the thylakoid membranes of the chloroplasts and require sunlight. Here’s how they work:\n\n- **Photon Absorption**: Chlorophyll, the green pigment in chloroplasts, absorbs sunlight, primarily in the blue and red wavelengths. This energy excites electrons in the chlorophyll molecules.\n\n- **Water Splitting (Photolysis)**: The absorbed energy is used to split water molecules (H₂O) into oxygen (O₂), protons (H⁺), and electrons. This reaction releases oxygen as a byproduct.\n\n- **Electron Transport Chain**: The excited electrons are 

In [130]:
# Not physics/math/history/CS/biology -> should be routed to DEFAULT
chain.run("Give me a recipe for a quick vegetarian pasta.")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


None: {'input': 'Give me a recipe for a quick vegetarian pasta.'}
> Finished chain.


"Here's a simple and quick vegetarian pasta recipe that you can whip up in about 20 minutes!\n\n### Quick Vegetarian Pasta\n\n#### Ingredients:\n- 8 oz (225 g) pasta (spaghetti, penne, or your choice)\n- 2 tablespoons olive oil\n- 3 cloves garlic, minced\n- 1 cup cherry tomatoes, halved\n- 1 cup spinach (fresh or frozen)\n- 1/2 teaspoon red pepper flakes (optional)\n- Salt and pepper to taste\n- 1/4 cup grated Parmesan cheese (optional, for serving)\n- Fresh basil or parsley for garnish (optional)\n\n#### Instructions:\n\n1. **Cook the Pasta:**\n   - Bring a large pot of salted water to a boil. Add the pasta and cook according to package instructions until al dente. Reserve about 1/2 cup of pasta water, then drain the pasta.\n\n2. **Sauté the Vegetables:**\n   - In a large skillet, heat the olive oil over medium heat. Add the minced garlic and sauté for about 30 seconds until fragrant.\n   - Add the halved cherry tomatoes and cook for about 3-4 minutes until they start to soften.\n   -